# Vowl Usage Patterns Demo

This notebook demonstrates the different methods of using the **vowl** library for data quality validation using [ODCS](https://github.com/bitol-io/open-data-contract-standard) data contracts.

## Table of Contents

1. [Setup](#1-setup)
2. [Local DataFrame (Pandas)](#2-local-dataframe-pandas)
3. [Local DataFrame (Polars)](#3-local-dataframe-polars)
4. [Ibis Connection (DuckDB)](#4-ibis-connection-duckdb)
5. [Explicit Adapter with Filter Conditions](#5-explicit-adapter-with-filter-conditions)
6. [Multi-Source Validation](#6-multi-source-validation)
7. [Working with ValidationResult](#7-working-with-validationresult)
8. [Database Connections (Testcontainers)](#8-database-connections-testcontainers)
   - [PostgreSQL](#81-postgresql)
   - [MySQL](#82-mysql)

# 1. Setup

In [ ]:
from pathlib import Path

# Resolve paths relative to the repo root
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "examples":
    REPO_ROOT = REPO_ROOT.parent  # examples -> repo root

# Single-source: HDB Resale dataset
HDB_DIR = REPO_ROOT / "tests" / "hdb_resale"
HDB_CSV = HDB_DIR / "HDBResale.csv"
HDB_CONTRACT = HDB_DIR / "hdb_resale_simple.yaml" # Switch out for "hdb_resale.yaml" to test a more complex contract

# Multi-source: PSD Employee dataset (payroll + employee list)
PSD_DIR = REPO_ROOT / "tests" / "psd_employee"
PSD_PAYROLL_CSV = PSD_DIR / "PSD_demo_employee_payroll.csv"
PSD_EMPLOYEE_LIST_CSV = PSD_DIR / "PSD_demo_employee_list.csv"
PSD_CONTRACT = PSD_DIR / "PSD_employee_payroll_datacontract.yaml"

print(f"HDB CSV exists:          {HDB_CSV.exists()}")
print(f"HDB contract exists:     {HDB_CONTRACT.exists()}")
print(f"PSD payroll CSV exists:  {PSD_PAYROLL_CSV.exists()}")
print(f"PSD emp list CSV exists: {PSD_EMPLOYEE_LIST_CSV.exists()}")
print(f"PSD contract exists:     {PSD_CONTRACT.exists()}")

HDB CSV exists:          True
HDB contract exists:     True
PSD payroll CSV exists:  True
PSD emp list CSV exists: True
PSD contract exists:     True


<a id="2-local-dataframe-pandas"></a>
## 2. Local DataFrame (Pandas)

The simplest usage pattern — load a CSV into a pandas DataFrame and validate it against a contract.

In [2]:
import pandas as pd
from vowl import validate_data

df = pd.read_csv(HDB_CSV)
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
df.head()

Loaded 201879 rows, 11 columns


/var/folders/tg/0hbw1fbs5q1_5kdtp36klrtw0000gn/T/ipykernel_87629/377769082.py:4: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(HDB_CSV)


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0


In [3]:
result = validate_data(contract=str(HDB_CONTRACT), df=df)
result.display_full_report()

/Users/jamestth/DEP/DCP/dqmk/src/vowl/validation/runner.py:72: UserWarning: Arrow type conversion failed, loading problematic columns as strings: lease_commence_date. Type validation will occur in DQ checks. Original error: ("Expected bytes, got a 'int' object", 'Conversion failed for column lease_commence_date with type object')
  resolved[schema_name] = mapper.get_adapter(adapter_input, schema_name)




=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           c11443ee-542f-4442-b28d-2d224342be37
   Schemas:               hdb_resale_prices

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       17 / 20 (85.0%)

   hdb_resale_prices:
     Overall:
       Checks Pass Rate:       17 / 20 (85.0%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       17 / 20 (85.0%)
       ERRORED Checks:         0
       Unique Passed Rows:     201,863 / 201,879 (99.9%)
     Multi Table:
       Checks Pass Rate:       0 / 0 (N/A)
       ERRORED Checks:         0
       Non-unique Failed Rows: 0


 CHECK RESULTS
+-----------------------------------------+---------------------------------------+-------------------+--------+---------------+---------------+--------+----------------+
| check_id                                | Target                                | tables_in_query   | status | operator      | expected      | actua

ValidationResult(passed=False, checks=20, passed_checks=17, failed_checks=3)

<a id="3-local-dataframe-polars"></a>
## 3. Local DataFrame (Polars)

vowl works with any [Narwhals-compatible](https://github.com/narwhals-dev/narwhals) DataFrame. Here we use Polars.

In [4]:
import polars as pl

polars_df = pl.read_csv(HDB_CSV, infer_schema_length=9999999) # To allow mixed type data to be detected in this demo example
print(f"Loaded {len(polars_df)} rows via Polars")

result = validate_data(contract=str(HDB_CONTRACT), df=polars_df)
result.display_full_report()

Loaded 201879 rows via Polars


=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           c11443ee-542f-4442-b28d-2d224342be37
   Schemas:               hdb_resale_prices

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       17 / 20 (85.0%)

   hdb_resale_prices:
     Overall:
       Checks Pass Rate:       17 / 20 (85.0%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       17 / 20 (85.0%)
       ERRORED Checks:         0
       Unique Passed Rows:     201,863 / 201,879 (99.9%)
     Multi Table:
       Checks Pass Rate:       0 / 0 (N/A)
       ERRORED Checks:         0
       Non-unique Failed Rows: 0


 CHECK RESULTS
+-----------------------------------------+---------------------------------------+-------------------+--------+---------------+---------------+--------+----------------+
| check_id                                | Target                                | tables_in_query   | status | operato

ValidationResult(passed=False, checks=20, passed_checks=17, failed_checks=3)

<a id="4-ibis-connection-duckdb "></a>
## 4. Ibis Connection (DuckDB) - Client-Side

Use an `IbisAdapter` to run SQL checks server-side through any of the [20+ Ibis backends](https://github.com/ibis-project/ibis) (PostgreSQL, Snowflake, BigQuery, DuckDB, etc.). Here we use DuckDB in-memory by materialising the table before validating data.

In [5]:
import ibis
from vowl import validate_data
from vowl.adapters import IbisAdapter

# Create an in-memory DuckDB connection and register the data
con = ibis.duckdb.connect()
hdb_df = pd.read_csv(HDB_CSV)
hdb_df["lease_commence_date"] = hdb_df["lease_commence_date"].astype(str) # To allow loading mixed type data to be detected in this demo example
con.create_table("hdb_resale_prices", hdb_df)

# Validate using the IbisAdapter — SQL checks run server-side
adapter = IbisAdapter(con)
result = validate_data(contract=str(HDB_CONTRACT), adapter=adapter)
result.display_full_report()

/var/folders/tg/0hbw1fbs5q1_5kdtp36klrtw0000gn/T/ipykernel_87629/3474147894.py:7: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  hdb_df = pd.read_csv(HDB_CSV)




=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           c11443ee-542f-4442-b28d-2d224342be37
   Schemas:               hdb_resale_prices

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       17 / 20 (85.0%)

   hdb_resale_prices:
     Overall:
       Checks Pass Rate:       17 / 20 (85.0%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       17 / 20 (85.0%)
       ERRORED Checks:         0
       Unique Passed Rows:     201,862 / 201,879 (99.9%)
     Multi Table:
       Checks Pass Rate:       0 / 0 (N/A)
       ERRORED Checks:         0
       Non-unique Failed Rows: 0


 CHECK RESULTS
+-----------------------------------------+---------------------------------------+-------------------+--------+---------------+---------------+--------+----------------+
| check_id                                | Target                                | tables_in_query   | status | operator      | expected      | actua

ValidationResult(passed=False, checks=20, passed_checks=17, failed_checks=3)

<a id="5-explicit-adapter-with-filter-conditions"></a>
## 5. Explicit Adapter with Filter Conditions

Filter conditions let you validate only a subset of your data — useful for incremental validation of new records. Supports wildcard patterns (`*`, `?`, `[seq]`).

In [6]:
from vowl.adapters import IbisAdapter, FilterCondition

con = ibis.duckdb.connect()
hdb_df = pd.read_csv(HDB_CSV)
hdb_df["lease_commence_date"] = hdb_df["lease_commence_date"].astype(str) # To allow loading mixed type data to be detected in this demo example
con.create_table("hdb_resale_prices", hdb_df)

# Dict-style filter: only validate rows from 2024 onwards
adapter = IbisAdapter(
    con,
    filter_conditions={
        "hdb_resale_prices": {
            "field": "month",
            "operator": ">=",
            "value": "2024-01",
        }
    },
)

result = validate_data(contract=str(HDB_CONTRACT), adapter=adapter)
result.print_summary()

/var/folders/tg/0hbw1fbs5q1_5kdtp36klrtw0000gn/T/ipykernel_87629/1973932249.py:4: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  hdb_df = pd.read_csv(HDB_CSV)




=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           c11443ee-542f-4442-b28d-2d224342be37
   Schemas:               hdb_resale_prices

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       19 / 20 (95.0%)

   hdb_resale_prices:
     Overall:
       Checks Pass Rate:       19 / 20 (95.0%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       19 / 20 (95.0%)
       ERRORED Checks:         0
       Unique Passed Rows:     32,727 / 32,729 (99.9%)
     Multi Table:
       Checks Pass Rate:       0 / 0 (N/A)
       ERRORED Checks:         0
       Non-unique Failed Rows: 0


 CHECK RESULTS
+-----------------------------------------+---------------------------------------+-------------------+--------+---------------+---------------+--------+----------------+
| check_id                                | Target                                | tables_in_query   | status | operator      | expected      | actual 

ValidationResult(passed=False, checks=20, passed_checks=19, failed_checks=1)

In [7]:
# Multiple filter conditions on the same table (combined with AND)
adapter = IbisAdapter(
    con,
    filter_conditions={
        "hdb_resale_prices": [
            FilterCondition(field="month", operator=">=", value="2024-01"),
            FilterCondition(field="town", operator="=", value="ANG MO KIO"),
        ]
    },
)

result = validate_data(contract=str(HDB_CONTRACT), adapter=adapter)
result.print_summary()



=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           c11443ee-542f-4442-b28d-2d224342be37
   Schemas:               hdb_resale_prices

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       20 / 20 (100.0%)

   hdb_resale_prices:
     Overall:
       Checks Pass Rate:       20 / 20 (100.0%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       20 / 20 (100.0%)
       ERRORED Checks:         0
       Unique Passed Rows:     1,264 / 1,264 (100.0%)
     Multi Table:
       Checks Pass Rate:       0 / 0 (N/A)
       ERRORED Checks:         0
       Non-unique Failed Rows: 0


 CHECK RESULTS
+-----------------------------------------+---------------------------------------+-------------------+--------+---------------+---------------+--------+----------------+
| check_id                                | Target                                | tables_in_query   | status | operator      | expected      | actua

ValidationResult(passed=True, checks=20, passed_checks=20, failed_checks=0)

<a id="6-multi-source-validation"></a>
## 6. Multi-Source Validation

Validate across tables that live in different source systems. The PSD Employee contract has two schemas — `PSD_demo_employee_payroll` and `PSD_demo_employee_list` — with cross-table referential integrity checks (e.g. "every payroll employee_id must exist in the master list").

> **Note:** Multi-adapter usage loads data into a local DuckDB instance before running quality checks. Ensure the local machine can handle the combined data size.

In [8]:
import warnings
import ibis
from vowl import validate_data
from vowl.adapters import IbisAdapter

# Load each table into its own connection (simulating different source systems)
payroll_df = pd.read_csv(PSD_PAYROLL_CSV)
employee_list_df = pd.read_csv(PSD_EMPLOYEE_LIST_CSV)

con_payroll = ibis.duckdb.connect()
con_payroll.create_table("PSD_demo_employee_payroll", payroll_df)

con_employees = ibis.duckdb.connect()
con_employees.create_table("PSD_demo_employee_list", employee_list_df)

# Map each schema name to its adapter
adapters = {
    "PSD_demo_employee_payroll": IbisAdapter(con_payroll),
    "PSD_demo_employee_list": IbisAdapter(con_employees),
}

result = validate_data(contract=str(PSD_CONTRACT), adapters=adapters)
result.display_full_report()



=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           b6376e57-aa90-41eb-90d0-c72dca7567e6
   Schemas:               PSD_demo_employee_payroll, PSD_demo_employee_list

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       70 / 76 (92.1%)

   PSD_demo_employee_payroll:
     Overall:
       Checks Pass Rate:       48 / 53 (90.5%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       48 / 51 (94.1%)
       ERRORED Checks:         0
       Unique Passed Rows:     0 / 3 (0.0%)
     Multi Table:
       Checks Pass Rate:       0 / 2 (0.0%)
       ERRORED Checks:         0
       Non-unique Failed Rows: 3

   PSD_demo_employee_list:
     Overall:
       Checks Pass Rate:       22 / 23 (95.6%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       22 / 23 (95.6%)
       ERRORED Checks:         0
       Unique Passed Rows:     0 / 2 (0.0%)
     Multi Table:
       Checks Pass Rate:   

ValidationResult(passed=False, checks=76, passed_checks=70, failed_checks=6)

In [9]:
result.save(output_dir=".", prefix="vowl_demo_multi_table_results")


Results saved:
   - vowl_demo_multi_table_results_check_results.csv
   - vowl_demo_multi_table_results_PSD_demo_employee_list.csv
   - vowl_demo_multi_table_results_PSD_demo_employee_list_PSD_demo_employee_payroll.csv
   - vowl_demo_multi_table_results_PSD_demo_employee_payroll.csv
   - vowl_demo_multi_table_results_summary.json


ValidationResult(passed=False, checks=76, passed_checks=70, failed_checks=6)

### Single adapter expanding to all schemas

When all tables live on the same connection, pass a single `adapter` — vowl will reuse it across all schemas in the contract.

In [10]:
# Both tables on the same DuckDB connection
con = ibis.duckdb.connect()
con.create_table("PSD_demo_employee_payroll", payroll_df)
con.create_table("PSD_demo_employee_list", employee_list_df)

adapter = IbisAdapter(con)

# vowl warns that a single adapter is being expanded to multiple schemas
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    result = validate_data(contract=str(PSD_CONTRACT), adapter=adapter)

result.display_full_report()



=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           b6376e57-aa90-41eb-90d0-c72dca7567e6
   Schemas:               PSD_demo_employee_payroll, PSD_demo_employee_list

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       70 / 76 (92.1%)

   PSD_demo_employee_payroll:
     Overall:
       Checks Pass Rate:       48 / 53 (90.5%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       48 / 51 (94.1%)
       ERRORED Checks:         0
       Unique Passed Rows:     0 / 3 (0.0%)
     Multi Table:
       Checks Pass Rate:       0 / 2 (0.0%)
       ERRORED Checks:         0
       Non-unique Failed Rows: 3

   PSD_demo_employee_list:
     Overall:
       Checks Pass Rate:       22 / 23 (95.6%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       22 / 23 (95.6%)
       ERRORED Checks:         0
       Unique Passed Rows:     0 / 2 (0.0%)
     Multi Table:
       Checks Pass Rate:   

ValidationResult(passed=False, checks=76, passed_checks=70, failed_checks=6)

<a id="7-working-with-validationresult"></a>
## 7. Working with ValidationResult

The `ValidationResult` object provides several ways to inspect and export results.

In [11]:
# Run a validation to work with
result = validate_data(contract=str(HDB_CONTRACT), df=pd.read_csv(HDB_CSV))

# Check overall pass/fail
print(f"All checks passed: {result.passed}")

# Print just the summary (without failed rows)
result.print_summary()

/var/folders/tg/0hbw1fbs5q1_5kdtp36klrtw0000gn/T/ipykernel_87629/2550493078.py:2: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  result = validate_data(contract=str(HDB_CONTRACT), df=pd.read_csv(HDB_CSV))
/Users/jamestth/DEP/DCP/dqmk/src/vowl/validation/runner.py:72: UserWarning: Arrow type conversion failed, loading problematic columns as strings: lease_commence_date. Type validation will occur in DQ checks. Original error: ("Expected bytes, got a 'int' object", 'Conversion failed for column lease_commence_date with type object')
  resolved[schema_name] = mapper.get_adapter(adapter_input, schema_name)


All checks passed: False


=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           c11443ee-542f-4442-b28d-2d224342be37
   Schemas:               hdb_resale_prices

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       17 / 20 (85.0%)

   hdb_resale_prices:
     Overall:
       Checks Pass Rate:       17 / 20 (85.0%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       17 / 20 (85.0%)
       ERRORED Checks:         0
       Unique Passed Rows:     201,863 / 201,879 (99.9%)
     Multi Table:
       Checks Pass Rate:       0 / 0 (N/A)
       ERRORED Checks:         0
       Non-unique Failed Rows: 0


 CHECK RESULTS
+-----------------------------------------+---------------------------------------+-------------------+--------+---------------+---------------+--------+----------------+
| check_id                                | Target                                | tables_in_query   | status | operator    

ValidationResult(passed=False, checks=20, passed_checks=17, failed_checks=3)

### Check Results DataFrame

`get_check_results_df()` returns a single DataFrame with one row per check — status, expected/actual values, execution time, dimension, and more.

In [12]:
# Full check results as a DataFrame (one row per check)
check_results_df = result.get_check_results_df()
check_results_df.to_pandas()

,check_name,target,schema,engine,type,dimension,description,status,severity,operator,...,failed_rows_count,aggregation_type,message,rule,tables_in_query,check_path,check_ref_type,logical_type,is_generated,execution_time_ms
0,month_column_exists_check,hdb_resale_prices.month,hdb_resale_prices,sql,sql,conformity,Column 'month' must exist in 'hdb_resale_prices',PASSED,None,mustBe,...,0,count,,"SELECT COUNT(*) FROM (SELECT ""month"" FROM ""hdb...",['hdb_resale_prices'],$.schema[0].properties[0].name,DeclaredColumnExistsCheckReference,string,True,1.680250
1,month_logical_type_check,hdb_resale_prices.month,hdb_resale_prices,sql,sql,conformity,Values in 'month' must be valid string,PASSED,None,mustBe,...,0,count,,"SELECT COUNT(*) FROM ""hdb_resale_prices"" WHERE...",['hdb_resale_prices'],$.schema[0].properties[0].logicalType,LogicalTypeCheckReference,string,True,3.077750
2,town_column_exists_check,hdb_resale_prices.town,hdb_resale_prices,sql,sql,conformity,Column 'town' must exist in 'hdb_resale_prices',PASSED,None,mustBe,...,0,count,,"SELECT COUNT(*) FROM (SELECT ""town"" FROM ""hdb_...",['hdb_resale_prices'],$.schema[0].properties[1].name,DeclaredColumnExistsCheckReference,None,True,1.502625
3,flat_type_column_exists_check,hdb_resale_prices.flat_type,hdb_resale_prices,sql,sql,conformity,Column 'flat_type' must exist in 'hdb_resale_p...,PASSED,None,mustBe,...,0,count,,"SELECT COUNT(*) FROM (SELECT ""flat_type"" FROM ...",['hdb_resale_prices'],$.schema[0].properties[2].name,DeclaredColumnExistsCheckReference,None,True,1.473084
4,block_column_exists_check,hdb_resale_prices.block,hdb_resale_prices,sql,sql,conformity,Column 'block' must exist in 'hdb_resale_prices',PASSED,None,mustBe,...,0,count,,"SELECT COUNT(*) FROM (SELECT ""block"" FROM ""hdb...",['hdb_resale_prices'],$.schema[0].properties[3].name,DeclaredColumnExistsCheckReference,None,True,1.447167
5,street_name_column_exists_check,hdb_resale_prices.street_name,hdb_resale_prices,sql,sql,conformity,Column 'street_name' must exist in 'hdb_resale...,PASSED,None,mustBe,...,0,count,,"SELECT COUNT(*) FROM (SELECT ""street_name"" FRO...",['hdb_resale_prices'],$.schema[0].properties[4].name,DeclaredColumnExistsCheckReference,None,True,1.614875
6,storey_range_column_exists_check,hdb_resale_prices.storey_range,hdb_resale_prices,sql,sql,conformity,Column 'storey_range' must exist in 'hdb_resal...,PASSED,None,mustBe,...,0,count,,"SELECT COUNT(*) FROM (SELECT ""storey_range"" FR...",['hdb_resale_prices'],$.schema[0].properties[5].name,DeclaredColumnExistsCheckReference,None,True,1.517625
7,floor_area_sqm_column_exists_check,hdb_resale_prices.floor_area_sqm,hdb_resale_prices,sql,sql,conformity,Column 'floor_area_sqm' must exist in 'hdb_res...,PASSED,None,mustBe,...,0,count,,"SELECT COUNT(*) FROM (SELECT ""floor_area_sqm"" ...",['hdb_resale_prices'],$.schema[0].properties[6].name,DeclaredColumnExistsCheckReference,None,True,1.683333
8,flat_model_column_exists_check,hdb_resale_prices.flat_model,hdb_resale_prices,sql,sql,conformity,Column 'flat_model' must exist in 'hdb_resale_...,PASSED,None,mustBe,...,0,count,,"SELECT COUNT(*) FROM (SELECT ""flat_model"" FROM...",['hdb_resale_prices'],$.schema[0].properties[7].name,DeclaredColumnExistsCheckReference,None,True,1.513542
9,lease_commence_date_column_exists_check,hdb_resale_prices.lease_commence_date,hdb_resale_prices,sql,sql,conformity,Column 'lease_commence_date' must exist in 'hd...,PASSED,None,mustBe,...,0,count,,"SELECT COUNT(*) FROM (SELECT ""lease_commence_d...",['hdb_resale_prices'],$.schema[0].properties[8].name,DeclaredColumnExistsCheckReference,None,True,1.506667


### Failed Rows (per check)

`get_output_dfs()` returns a dict of `{check_id: DataFrame}` — one entry per failed check, containing the actual rows that failed.

In [13]:
# Failed rows grouped by check — each key is "schema::check_name"
output_dfs = result.get_output_dfs()
for check_id, failed_df in output_dfs.items():
    if len(failed_df) == 0:
        continue  # Skip checks with no failed rows
    print(f"\n{'='*60}")
    print(f"Check: {check_id}  ({len(failed_df)} failed rows)")
    print(f"{'='*60}")
    display(failed_df.to_pandas().head())


Check: hdb_resale_prices::Month  (2 failed rows)


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,check_id,tables_in_query
0,2017-jan,BEDOK,5 ROOM,21,CHAI CHEE RD,07 TO 09,130.0,Adjoined flat,1972,54 years 06 months,530000.0,Month,hdb_resale_prices
1,2017-jan,BISHAN,3 ROOM,105,BISHAN ST 12,04 TO 06,4.0,Simplified,1985,67 years 11 months,395000.0,Month,hdb_resale_prices



Check: hdb_resale_prices::Year  (2 failed rows)


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,check_id,tables_in_query
0,2017-01,ANG MO KIO,3 ROOM,219,ANG MO KIO AVE 1,07 TO 09,67.0,New Generation,1977.0,59 years 06 months,297000.0,Year,hdb_resale_prices
1,2017-01,ANG MO KIO,3 ROOM,211,ANG MO KIO AVE 3,01 TO 03,67.0,New Generation,abc,59 years 03 months,325000.0,Year,hdb_resale_prices



Check: hdb_resale_prices::floor_area_must_be_less_than_200  (12 failed rows)


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,check_id,tables_in_query
0,2017-06,KALLANG/WHAMPOA,3 ROOM,38,JLN BAHAGIA,01 TO 03,215.0,Terrace,1972,54 years 01 month,830000.0,floor_area_must_be_less_than_200,hdb_resale_prices
1,2017-09,CHOA CHU KANG,EXECUTIVE,641,CHOA CHU KANG ST 64,16 TO 18,215.0,Premium Maisonette,1998,79 years 04 months,888000.0,floor_area_must_be_less_than_200,hdb_resale_prices
2,2017-12,KALLANG/WHAMPOA,3 ROOM,65,JLN MA'MOR,01 TO 03,249.0,Terrace,1972,53 years 07 months,1053888.0,floor_area_must_be_less_than_200,hdb_resale_prices
3,2018-01,CHOA CHU KANG,EXECUTIVE,639,CHOA CHU KANG ST 64,10 TO 12,215.0,Premium Maisonette,1998,79 years,900000.0,floor_area_must_be_less_than_200,hdb_resale_prices
4,2018-09,KALLANG/WHAMPOA,3 ROOM,41,JLN BAHAGIA,01 TO 03,237.0,Terrace,1972,52 years 10 months,1185000.0,floor_area_must_be_less_than_200,hdb_resale_prices


### Consolidated Failed Rows (per table)

`get_consolidated_output_dfs()` deduplicates failed rows across all checks and groups them by table. Useful when multiple checks flag the same row.

In [14]:
# Consolidated: deduplicated failed rows grouped by table
consolidated = result.get_consolidated_output_dfs()
for table_name, consolidated_df in consolidated.items():
    print(f"\n{'='*60}")
    print(f"Table: {table_name}  ({len(consolidated_df)} unique failed rows)")
    print(f"{'='*60}")
    display(consolidated_df.to_pandas().head())


Table: hdb_resale_prices  (16 unique failed rows)


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,check_ids,tables_in_query
0,2017-jan,BEDOK,5 ROOM,21,CHAI CHEE RD,07 TO 09,130.0,Adjoined flat,1972,54 years 06 months,530000.0,Month,hdb_resale_prices
1,2017-jan,BISHAN,3 ROOM,105,BISHAN ST 12,04 TO 06,4.0,Simplified,1985,67 years 11 months,395000.0,Month,hdb_resale_prices
2,2017-01,ANG MO KIO,3 ROOM,219,ANG MO KIO AVE 1,07 TO 09,67.0,New Generation,1977.0,59 years 06 months,297000.0,Year,hdb_resale_prices
3,2017-01,ANG MO KIO,3 ROOM,211,ANG MO KIO AVE 3,01 TO 03,67.0,New Generation,abc,59 years 03 months,325000.0,Year,hdb_resale_prices
4,2017-06,KALLANG/WHAMPOA,3 ROOM,38,JLN BAHAGIA,01 TO 03,215.0,Terrace,1972,54 years 01 month,830000.0,floor_area_must_be_less_than_200,hdb_resale_prices


In [15]:
# Save results to disk (CSV + JSON summary)
result.save(output_dir=".", prefix="vowl_demo_HDB_results")


Results saved:
   - vowl_demo_HDB_results_check_results.csv
   - vowl_demo_HDB_results_hdb_resale_prices.csv
   - vowl_demo_HDB_results_summary.json


ValidationResult(passed=False, checks=20, passed_checks=17, failed_checks=3)

<a id="8-database-connections-testcontainers"></a>
## 8. Database Connections (Testcontainers)

The examples below use [testcontainers](https://testcontainers-python.readthedocs.io/) to spin up real database instances in Docker. This demonstrates how vowl works with actual database backends for server side validation.

**Requirements:**
- Docker must be running
- Dev dependencies: `uv sync --dev`

<a id="81-postgresql"></a>
### 8.1 PostgreSQL

In [16]:
import os
from pathlib import Path as _Path

import ibis
import pandas as pd
from testcontainers.postgres import PostgresContainer
from vowl import validate_data
from vowl.adapters import IbisAdapter

# Configure Docker Desktop socket path (macOS)
docker_sock = _Path.home() / ".docker" / "run" / "docker.sock"
if docker_sock.exists() and "DOCKER_HOST" not in os.environ:
    os.environ["DOCKER_HOST"] = f"unix://{docker_sock}"

# Disable Ryuk reaper for Docker Desktop compatibility
os.environ.setdefault("TESTCONTAINERS_RYUK_DISABLED", "true")

postgres = PostgresContainer("postgres:15-alpine")
postgres.start()

# Connect via Ibis
pg_con = ibis.postgres.connect(
    host=postgres.get_container_host_ip(),
    port=postgres.get_exposed_port(5432),
    user=postgres.username,
    password=postgres.password,
    database=postgres.dbname,
)

# Load HDB data with proper types for PostgreSQL
hdb_df = pd.read_csv(HDB_CSV).head(200)
for col in ("floor_area_sqm", "lease_commence_date", "resale_price"):
    hdb_df[col] = pd.to_numeric(hdb_df[col], errors="coerce").astype("Int64")

pg_con.raw_sql("""
    CREATE TABLE hdb_resale_prices (
        month TEXT, town TEXT, flat_type TEXT, block TEXT,
        street_name TEXT, storey_range TEXT, floor_area_sqm INTEGER,
        flat_model TEXT, lease_commence_date INTEGER, remaining_lease TEXT,
        resale_price INTEGER
    )
""")
pg_con.insert("hdb_resale_prices", hdb_df)

adapter = IbisAdapter(pg_con)
result = validate_data(contract=str(HDB_CONTRACT), adapter=adapter)
result.display_full_report()

# Clean up
postgres.stop()

/var/folders/tg/0hbw1fbs5q1_5kdtp36klrtw0000gn/T/ipykernel_87629/5764842.py:31: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  hdb_df = pd.read_csv(HDB_CSV).head(200)




=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           c11443ee-542f-4442-b28d-2d224342be37
   Schemas:               hdb_resale_prices

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       19 / 20 (95.0%)

   hdb_resale_prices:
     Overall:
       Checks Pass Rate:       19 / 20 (95.0%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       19 / 20 (95.0%)
       ERRORED Checks:         0
       Unique Passed Rows:     198 / 200 (99.0%)
     Multi Table:
       Checks Pass Rate:       0 / 0 (N/A)
       ERRORED Checks:         0
       Non-unique Failed Rows: 0


 CHECK RESULTS
+-----------------------------------------+---------------------------------------+-------------------+--------+---------------+---------------+--------+----------------+
| check_id                                | Target                                | tables_in_query   | status | operator      | expected      | actual | exec

<a id="82-mysql"></a>
### 8.2 MySQL

> **Note:** For MySQL, select the database when creating the connection (via `database=` param or in the connection URI). vowl does not issue `USE database` — it runs read-only `SELECT` queries against the active database.

In [17]:
from testcontainers.mysql import MySqlContainer

mysql = MySqlContainer("mysql:8.0")
mysql.start()

mysql_con = ibis.mysql.connect(
    host=mysql.get_container_host_ip(),
    port=int(mysql.get_exposed_port(3306)),
    user=mysql.username,
    password=mysql.password,
    database=mysql.dbname,
)

# Load data — MySQL requires TEXT types for string columns
hdb_df = pd.read_csv(HDB_CSV).head(200).astype(str)

mysql_con.raw_sql("""
    CREATE TABLE hdb_resale_prices (
        month TEXT, town TEXT, flat_type TEXT, block TEXT,
        street_name TEXT, storey_range TEXT, floor_area_sqm TEXT,
        flat_model TEXT, lease_commence_date TEXT, remaining_lease TEXT,
        resale_price TEXT
    )
""")
mysql_con.insert("hdb_resale_prices", hdb_df)

adapter = IbisAdapter(mysql_con)
result = validate_data(contract=str(HDB_CONTRACT), adapter=adapter)
result.display_full_report()

# Clean up
mysql.stop()

/var/folders/tg/0hbw1fbs5q1_5kdtp36klrtw0000gn/T/ipykernel_87629/3107984563.py:15: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  hdb_df = pd.read_csv(HDB_CSV).head(200).astype(str)




=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           c11443ee-542f-4442-b28d-2d224342be37
   Schemas:               hdb_resale_prices

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       18 / 20 (90.0%)

   hdb_resale_prices:
     Overall:
       Checks Pass Rate:       18 / 20 (90.0%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       18 / 20 (90.0%)
       ERRORED Checks:         0
       Unique Passed Rows:     195 / 200 (97.5%)
     Multi Table:
       Checks Pass Rate:       0 / 0 (N/A)
       ERRORED Checks:         0
       Non-unique Failed Rows: 0


 CHECK RESULTS
+-----------------------------------------+---------------------------------------+-------------------+--------+---------------+---------------+--------+----------------+
| check_id                                | Target                                | tables_in_query   | status | operator      | expected      | actual | exec